## Loading all the pdfs for rag

In [1]:
!pip install -qU langchain-community pypdf

In [2]:
from langchain_community.document_loaders import PyPDFLoader

pdf_urls = [
    "https://investor.sebi.gov.in/pdf/downloadable-documents/Financial%20Education%20Booklet%20-%20English.pdf",
    "https://a2ztaxcorp.net/wp-content/uploads/2025/09/CBIC-GST-Ready-Reckoner-indicating-updated-Central-Goods-and-Services-Tax-CGST-rates-on-goods.pdf",
]

all_documents = []
for url in pdf_urls:
    loader = PyPDFLoader(url)
    documents = loader.load()
    print(f"Loaded {len(documents)} pages from {url.split('/')[-1]}")
    all_documents.extend(documents)

/tmp/ipykernel_1782/3185070237.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 73 pages from Financial%20Education%20Booklet%20-%20English.pdf
Loaded 94 pages from CBIC-GST-Ready-Reckoner-indicating-updated-Central-Goods-and-Services-Tax-CGST-rates-on-goods.pdf


### Code Explanation: PDF Loading Loop

* **`all_documents = []`**
  Creates an empty list to store all PDF content.

* **`for url in pdf_urls:`**
  Loops through each PDF URL one by one.

* **`loader = PyPDFLoader(url)`**
  Creates a loader object for the current PDF.

* **`documents = loader.load()`**
  Reads the PDF and loads its content into `documents`.

* **`all_documents.extend(documents)`**
  Adds everything inside `documents` into `all_documents`.
  > **Note:** `extend()` means *"Take all items from documents and move/add them into all_documents."*

##Split, Embed, and Store

In [3]:
!pip install -qU langchain langchain-core langchain-community langchain-text-splitters \
    langchain-huggingface langchain-chroma pypdf nltk chromadb

In [6]:
#!pip install -U --quiet opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp-proto-grpc opentelemetry-exporter-otlp-proto-common opentelemetry-proto chromadb

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(all_documents)
print(f"Total chunks: {len(chunks)}")

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_store = Chroma(
    collection_name="finance_docs",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db"
)
vector_store.add_documents(documents=chunks)
print("Knowledge base ready!")

Total chunks: 504


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base ready!


### Code Explanation: Document Splitting and Vector Store Setup

* **Imports**
  Loads necessary tools from LangChain to break down text (`RecursiveCharacterTextSplitter`), convert text into mathematical vectors (`HuggingFaceEmbeddings`), and store those vectors in a database (`Chroma`).

* **`text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)`**
  Initializes a tool to break long documents into smaller pieces of up to 1000 characters. The `chunk_overlap=200` ensures that 200 characters are shared between consecutive chunks to maintain context at the cuts.

* **`chunks = text_splitter.split_documents(all_documents)`**
  Takes the list of full documents and splits them into the smaller chunks.

* **`print(f"Total chunks: {len(chunks)}")`**
  Outputs the total number of chunks created from the documents.

* **`embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")`**
  Loads a fast and lightweight pre-trained HuggingFace model (`all-MiniLM-L6-v2`). This model translates human-readable text into numerical vectors (embeddings) so the system can understand text meaning and similarity.

* **`vector_store = Chroma(...)`**
  Sets up a local Chroma vector database to store our data:
  * `collection_name="finance_docs"`: Names a specific folder/collection inside the database.
  * `embedding_function=embeddings`: Tells the database to use the MiniLM model when processing the text.
  * `persist_directory="./chroma_langchain_db"`: Saves the database to your local hard drive so you don't lose the data when you close the notebook.

* **`vector_store.add_documents(documents=chunks)`**
  Calculates the vector embeddings for all the split text chunks and saves both the original text and the vectors into the Chroma database.

* **`print("Knowledge base ready!")`**
  Prints a confirmation message indicating the vector database is fully loaded and ready to be queried.

##Converting to a Tool

In [5]:
from langchain.tools import tool

@tool
def search_finance_docs(question: str) -> str:
    """Search official Indian government financial documents for rules, regulations, tax rates, investment guidance, and financial education content."""
    results = vector_store.similarity_search(question, k=3)
    if not results:
        return "No relevant information found in the knowledge base."
    context = ""
    for doc in results:
        source = doc.metadata.get('source', 'Unknown')
        page = doc.metadata.get('page', 'N/A')
        context += f"Source: {source} (Page {page + 1})\n"
        context += f"Content: {doc.page_content}\n\n"
    return context

### Code Explanation: Creating a Custom Search Tool

* **`from langchain.tools import tool`**
  Imports the `@tool` decorator, which LangChain uses to convert a standard Python function into an tool that an AI agent can understand and use.

* **`@tool` and `def search_finance_docs(question: str) -> str:`**
  Defines the custom tool named `search_finance_docs`. It takes a `question` as a string input and returns a string as the output.

* **`"""Search official Indian government..."""`**
  This docstring is very important—it acts as instructions for the AI agent, telling it *when* and *why* it should use this specific tool to answer a user's question.

* **`results = vector_store.similarity_search(question, k=3)`**
  Takes the user's question, converts it into a vector, and searches the Chroma database (`vector_store`) for the top 3 (`k=3`) most similar text chunks.

* **`if not results: ...`**
  A safety check. If the database doesn't find any matching documents, it returns a clear message instead of crashing or hallucinating.

* **Formatting the `context` String (The `for` loop)**
  Loops through the top 3 results and builds a readable text block containing:
  * **`source`**: The origin of the document (retrieved from metadata).
  * **`page`**: The page number. It adds `1` (`page + 1`) because LangChain's PDF loader typically starts counting pages at `0`.
  * **`doc.page_content`**: The actual text retrieved from the document.

* **`return context`**
  Sends the formatted text block back to the AI agent so it can read the information and formulate a final answer for the user.

##Adding Live Market Data

In [6]:
@tool
def get_market_price(asset: str) -> str:
    """Get current market price for a financial asset in Indian Rupees. Pass one of: gold, silver."""
    symbols = {
        "gold": "XAU",
        "silver": "XAG",
    }
    symbol = symbols.get(asset.lower())
    if not symbol:
        return "Supported assets: gold, silver"

    url = f"https://api.gold-api.com/price/{symbol}/INR"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        price = data.get("price")
        if price:
            price_per_gram = price / 31.1035
            return f"{asset.title()}: ₹{price_per_gram:.2f} per gram (₹{price:.2f} per troy ounce). Source: Gold-API"
    return f"Could not fetch price for {asset}"

### Code Explanation: Live Market Price Tool

* **`@tool` and `def get_market_price(asset: str) -> str:`**
  Uses a LangChain decorator to turn a standard Python function into an AI-accessible tool. It expects an `asset` (like "gold" or "silver") and returns a string with the price.

* **`"""Get current market price for..."""`**
  This docstring is the instruction manual for the AI agent. It tells the LLM exactly when to use this tool and what inputs ("gold" or "silver") are acceptable.

* **`symbols = {"gold": "XAU", "silver": "XAG"}`**
  A dictionary that maps common asset names to their official financial ticker symbols (XAU for Gold, XAG for Silver).

* **`symbol = symbols.get(asset.lower())`**
  Takes the user's input, converts it to lowercase to avoid case-sensitivity issues, and looks up the corresponding ticker symbol.

* **`if not symbol: return "Supported assets: gold, silver"`**
  An error-handling safety net. If the user asks for an unsupported asset (like "platinum"), the tool politely rejects it instead of breaking.

* **`url = f"https://api.gold-api.com/price/{symbol}/INR"`**
  Constructs the specific API web address to fetch the live price of the requested asset in Indian Rupees (INR).

* **`response = requests.get(url)`**
  Makes an HTTP request to the internet to fetch the data from the constructed URL.

* **`if response.status_code == 200:` and `data = response.json()`**
  Checks if the web request was successful (Status `200` means OK). If so, it reads the data and converts it from JSON format into a Python dictionary.

* **`price = data.get("price")`**
  Extracts the exact price value from the API's response data.

* **`price_per_gram = price / 31.1035`**
  Converts the price from "per troy ounce" (the standard global market measurement) to "per gram". A troy ounce is strictly equivalent to exactly 31.1034768 grams, which is approximated here to 31.1035.

* **`return f"{asset.title()}: ₹{price_per_gram:.2f}..."`**
  Formats the final math calculations into a neat, human-readable string rounded to two decimal places (`.2f`), which the AI then presents to the user.

## The Full AI Advisor

In [7]:
!pip install -qU langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 kB 2.4 MB/s eta 0:00:00


In [8]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from google.colab import userdata

api_key = userdata.get('gemini_api_key')

model = init_chat_model(
    "google_genai:gemini-2.5-flash",
    api_key=api_key,
)

### Code Explanation: LLM Initialization and Security

* **Imports**
  * `init_chat_model`: A LangChain utility to easily initialize different chat models.
  * `create_agent`: Imported to build the AI agent (likely used in the next steps of your code).
  * `userdata`: A Google Colab specific library used to securely manage sensitive information like API keys.

* **`api_key = userdata.get('GEMINI_API_KEY')`**
  Securely retrieves your private Gemini API key from Google Colab's built-in "Secrets" tab. This ensures your key is not exposed or hardcoded directly in plain text within the notebook.

* **`model = init_chat_model(...)`**
  Initializes the Large Language Model (LLM) that will power your AI agent.
  * `"google_genai:gemini-2.5-flash"`: Specifies both the provider (`google_genai`) and the exact model version (`gemini-2.5-flash`) to use. The Flash model is optimized for fast and cost-effective reasoning.
  * `api_key=api_key`: Authenticates the model using the secure key retrieved in the previous step.

In [9]:
system_prompt = """You are a Personal Finance AI Advisor for Indian citizens.

You have access to these tools:
- search_finance_docs: Search official Indian government documents for GST rates,
  investment guidance, tax saving options, and financial education
- get_market_price: Get live market prices for gold and silver in Indian Rupees

Help the user by looking up the relevant data using your tools and giving clear,
specific answers with actual numbers and rates. Always mention the source of
your information. All monetary values should be in Indian Rupees (₹) unless
specified otherwise.
"""

agent = create_agent(
    model=model,
    tools=[search_finance_docs, get_market_price],
    system_prompt=system_prompt,
)

### Code Explanation: Assembling the AI Agent

* **`agent = create_agent(...)`**
  This function combines the language model, your custom tools, and the core instructions into a single, cohesive AI agent capable of reasoning, taking actions, and answering user queries.

* **`model=model`**
  Connects the agent to the "brain" you initialized earlier (the Gemini 2.5 Flash model). This handles the actual natural language understanding and generation.

* **`tools=[search_finance_docs, get_market_price]`**
  Equips the agent with the custom tools you built. By passing this list, the AI is granted permission to autonomously search your local finance PDF database or fetch live gold/silver prices from the internet whenever it decides those actions are necessary to answer a question.

* **`system_prompt=system_prompt`**
  Provides the agent with its foundational instructions (defined in your `system_prompt` variable). This dictates the agent's persona, guardrails, and exactly how it should behave and format its responses when interacting with users.

In [10]:
import requests

response = agent.invoke({
    "messages": [{"role": "user", "content": "What is the current gold price?"}]
})
print(response["messages"][-1].content)

response = agent.invoke({
    "messages": [{"role": "user", "content": "I want to buy a gold chain. What's the current gold price and how much GST will I pay?"}]
})
print(response["messages"][-1].content)

The current gold price is ₹13264.60 per gram (₹412575.36 per troy ounce). This information is sourced from Gold-API.
[{'type': 'text', 'text': 'The current market price of gold is ₹13264.60 per gram. (Source: Gold-API)\n\nI am unable to find the exact GST rate for gold in the available documents. The search results mention CGST rates of 2.5% for some goods, but gold is not explicitly listed. Generally, gold attracts a 3% GST on its value and 5% on making charges, but I need to confirm this with an official source. I will try to find this information for you.', 'extras': {'signature': 'CvsXARFNMg8XdcJonmo91hVZHj35UbCkjnbi+aOC4FopEvA0AXwGBUqsWNXJ2Xp6d03qIEbuA3QU1hfr5c8f6tC7f6AOpW5/anDuvicGuWQvWRjiM4hZFZxNb+JKzY0XzE1fPLnzc5do+85r6HAeqv0mtj4FXovI2Hx0X1yW6VsXfnH4vfvAec+4m82WGq3NY7eAH5fi7a7PLDyYkd+uv3pcsJCmhxeZQYOaCejOEPm5WaGRyEu0jB9NJILCmM694oEcflGs3VATY4BQmNJkO/BTsKg74bFyTn1T3V3OcfnEn+OhI7DNHlq7N0ibrseDEwP3IGABxLkQzZ0i8zs7qYQBa6aoNvelMGOJZThjmsIW2AYHVqhXGAT+ed96KTQUPk2xeAN0wcHr3XN8DjPkxKgE

## Building the UI

In [ ]:
!pip install -qU gradio

In [ ]:
import gradio as gr

def extract_text(content):
    """Handle both plain string and list-of-content-block responses."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "\n".join(
            block.get("text", "")
            for block in content
            if isinstance(block, dict) and block.get("type") == "text"
        )
    return str(content)

def finance_advisor(question):
    response = agent.invoke({"messages": [{"role": "user", "content": question}]})
    last_message = response["messages"][-1]
    return extract_text(last_message.content)

demo = gr.Interface(
    fn=finance_advisor,
    inputs=gr.Textbox(lines=2, placeholder="Ask a finance question...", label="Question"),
    outputs=gr.Textbox(lines=10, label="Answer"),
    title="Personal Finance AI Advisor",
    description="Ask about gold prices, silver prices, GST rates, tax saving options, and more.",
)
demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://971352ad03749d354c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://971352ad03749d354c.gradio.live


## Adding Voice Interaction

In [ ]:
!pip install -qU assemblyai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 685.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.4/88.4 kB 2.2 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
import assemblyai as aai

ASSEMBLYAI_API_KEY = userdata.get('ASSEMBLYAI_API_KEY')
aai.settings.api_key = ASSEMBLYAI_API_KEY

def speech_to_text(audio_path):
    """Converts an audio file to text using AssemblyAI."""
    transcriber = aai.Transcriber()
    config = aai.TranscriptionConfig(
        speech_models=["universal-3-5-pro", "universal-2"],
        language_detection=True,
        speaker_labels=True,
    )
    transcript = transcriber.transcribe(audio_path, config=config)
    return transcript.text if transcript.text else ""

In [ ]:
def finance_advisor_voice(audio):
    if audio is None:
        return "No audio recorded.", "Please record your question first."
    question = speech_to_text(audio)
    response = agent.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content
    return question, answer

In [ ]:
import gradio as gr

voice_demo = gr.Interface(
    fn=finance_advisor_voice,
    inputs=gr.Audio(sources=["microphone", "upload"], type="filepath", label="Ask your question"),
    outputs=[
        gr.Textbox(label="Your Question (transcribed)"),
        gr.Textbox(lines=10, label="Answer"),
    ],
    title="Personal Finance AI Advisor (Voice)",
    description="Speak your finance question and get a text response.",
)

voice_demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b81a14303ea156c2e5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://b81a14303ea156c2e5.gradio.live


## RAG Evaluation using RAGAS FRAMEWORK

RAGAS (Retrieval Augmented Generation Assessment) is an open-source library that automatically evaluates our RAG system. Instead of manually checking every answer, RAGAS uses an LLM to judge how well our system is working — both the retrieval (did we find the right documents?) and the generation (did we use them correctly?)


---




In [11]:
test_questions = [
    {
        "user_input": "What is the GST rate on laptops and computers?",
        "retrieved_contexts": ["Laptops and computers attract 18% GST as per the GST council classification."],
        "response": "Laptops and computers attract 18% GST.",
        "reference": "Laptops and computers attract 18% GST.",
    },
    {
        "user_input": "What are the tax saving options under Section 80C?",
        "retrieved_contexts": ["Section 80C of the Income Tax Act allows deductions through investments in life insurance, provident fund, ELSS mutual fund schemes, 5-year bank term deposits, NPS Tier 1 account, National Savings Certificate, and housing loan principal repayment."],
        "response": "Under Section 80C, you can save tax by investing in life insurance, provident fund, ELSS, 5-year FDs, NPS, NSC, and home loan principal repayment.",
        "reference": "Section 80C of the Income Tax Act allows deductions through investments in life insurance, provident fund, ELSS mutual fund schemes, 5-year bank term deposits, NPS Tier 1 account, National Savings Certificate, and housing loan principal repayment.",
    },
]

In [12]:
!pip install -q -U ragas langchain-google-genai langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.6/123.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.9/353.9 kB 18.7 MB/s eta 0:00:00


In [13]:
!pip install -q langchain-google-genai

In [26]:
!pip uninstall -y ragas langchain-community
!pip install -q ragas==0.3.9 "langchain-community<0.4" langchain-google-genai

Found existing installation: ragas 0.3.9
Uninstalling ragas-0.3.9:
  Successfully uninstalled ragas-0.3.9
Found existing installation: langchain-community 0.4.2
Uninstalling langchain-community-0.4.2:
  Successfully uninstalled langchain-community-0.4.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.9 MB/s eta 0:00:00


In [27]:
!pip show ragas langchain-community | grep -iE "^(Name|Version)"

Name: ragas
Version: 0.3.9
Name: langchain-community
Version: 0.3.31


In [28]:
from ragas.llms import LangchainLLMWrapper
print("ragas import OK")

ragas import OK


In [30]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import Faithfulness, ContextPrecision, ContextRecall
from ragas import evaluate, EvaluationDataset

# 1. Set API key (pull from Colab secrets, same pattern as your gemini_api_key secret)
os.environ["GOOGLE_API_KEY"] = userdata.get('gemini_api_key')

# 2. Setup LLM
evaluator_llm = LangchainLLMWrapper(
    ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        google_api_key=os.environ["GOOGLE_API_KEY"]
    )
)
print("LLM ready:", evaluator_llm)

# 3. Define your data
eval_data = [
    {
        "user_input": "What is the GST rate on laptops and computers?",
        "retrieved_contexts": ["Laptops and computers attract 18% GST as per the GST council classification."],
        "response": "Laptops and computers attract 18% GST.",
        "reference": "Laptops and computers attract 18% GST.",
    },
    {
        "user_input": "What are the tax saving options under Section 80C?",
        "retrieved_contexts": ["Section 80C of the Income Tax Act allows deductions through investments in life insurance, provident fund, ELSS mutual fund schemes, 5-year bank term deposits, NPS Tier 1 account, National Savings Certificate, and housing loan principal repayment."],
        "response": "Under Section 80C, you can save tax by investing in life insurance, provident fund, ELSS, 5-year FDs, NPS, NSC, and home loan principal repayment.",
        "reference": "Section 80C of the Income Tax Act allows deductions through investments in life insurance, provident fund, ELSS mutual fund schemes, 5-year bank term deposits, NPS Tier 1 account, National Savings Certificate, and housing loan principal repayment.",
    },
]

# 4. Create dataset
eval_dataset = EvaluationDataset.from_list(eval_data)
print(f"Dataset size: {len(eval_dataset.samples)}")

# 5. Define metrics
faithfulness = Faithfulness(llm=evaluator_llm)
context_precision = ContextPrecision(llm=evaluator_llm)
context_recall = ContextRecall(llm=evaluator_llm)

# 6. Evaluate
results = evaluate(
    dataset=eval_dataset,
    metrics=[faithfulness, context_precision, context_recall],
    llm=evaluator_llm,
)
print(results)

/tmp/ipykernel_1782/786769369.py:12: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(


LLM ready: LangchainLLMWrapper(langchain_llm=ChatGoogleGenerativeAI(...))
Dataset size: 2


Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]

{'faithfulness': 1.0000, 'context_precision': 1.0000, 'context_recall': 1.0000}
